In [3]:
!pip install scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 102.5 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 37.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 33.9 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Setup & Imports

In [4]:
import os
import pathlib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram

Dateien einlesen

In [ ]:

# Ordnerpfad anpassen
BASE_DIR = pathlib.Path("pfad/zum/projektordner")

# Dateiendungen definieren
EXTENSIONS = {".mel", ".src"}

files = [f for f in BASE_DIR.rglob("*") if f.suffix in EXTENSIONS]

print(f"Gefundene Dateien: {len(files)}")
files[:10]


Inhalt laden

In [ ]:
texts = []
for f in files:
    try:
        content = f.read_text(errors="ignore")
    except:
        content = ""
    texts.append(content)


Tokenisierung & TF‑IDF‑Vektorisierung

In [ ]:
vectorizer = TfidfVectorizer(
    token_pattern=r"[A-Za-z_]\w+",
    lowercase=True,
    max_df=0.9,
    min_df=1
)

X = vectorizer.fit_transform(texts)
X.shape

Ähnlichkeitsmatrix berechnen

In [ ]:
similarity = cosine_similarity(X)
similarity_df = pd.DataFrame(
    similarity, 
    index=[f.name for f in files],
    columns=[f.name for f in files]
)

similarity_df.head()

Ähnlichste Dateipaare ermitteln

In [ ]:
pairs = []
threshold = 0.80  # anpassbar

for i in range(len(files)):
    for j in range(i+1, len(files)):
        sim = similarity[i, j]
        if sim >= threshold:
            pairs.append((files[i].name, files[j].name, sim))

pairs_sorted = sorted(pairs, key=lambda x: -x[2])
pairs_sorted[:20]

Versionserkennung per Nearest Neighbor


In [ ]:
nbrs = NearestNeighbors(n_neighbors=3, metric="cosine").fit(X)
distances, indices = nbrs.kneighbors(X)

results = []
for idx, neighbors in enumerate(indices):
    base = files[idx].name
    for n_idx in neighbors[1:]:
        neighbor = files[n_idx].name
        sim = 1 - distances[idx][1]
        results.append((base, neighbor, sim))

results[:20]

Hierarchisches Clustering (Dendrogramm)

In [ ]:
Z = linkage(X.toarray(), method='ward')

plt.figure(figsize=(14, 6))
dendrogram(Z, labels=[f.name for f in files], leaf_rotation=90)
plt.title("Hierarchisches Clustering der Dateien")
plt.tight_layout()
plt.show()

Optional: Automatisches Gruppieren mit DBSCAN

In [ ]:
clustering = DBSCAN(eps=0.3, min_samples=2, metric="cosine").fit(X)
labels = clustering.labels_

cluster_df = pd.DataFrame({
    "file": [f.name for f in files],
    "cluster": labels
})

cluster_df.sort_values("cluster")